# FABLE Pyculator 2021 FreshForge Run

This notebook shows how to use FABLE Pyculator to derive the 2021 FABLE-C output-ref boundary, then use FreshForge to run the Modelwright generated-model workflow.

## Local artifacts

The source workbook stays under ignored `tmp/` paths. The generated Python model and run evidence are written under `tmp/generated-models/fable-2021/`.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

from IPython.display import Markdown, display

from fable_pyculator import (
    DEFAULT_2021_WORKBOOK_PATH,
    build_2021_notebook_spec,
    load_generated_model,
    run_notebook_loop,
)


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "fable_pyculator").exists():
            return candidate
    raise RuntimeError("Could not find the fable-pyculator repository root.")


repo_root = find_repo_root(Path.cwd().resolve())
workbook_path = repo_root / DEFAULT_2021_WORKBOOK_PATH
artifact_dir = repo_root / "tmp" / "generated-models" / "fable-2021"
artifact_dir.mkdir(parents=True, exist_ok=True)

output_refs_path = artifact_dir / "output_refs.json"
workflow_path = artifact_dir / "freshforge-modelwright-run-workflow.json"
contract_path = artifact_dir / "contract.json"
expressions_path = artifact_dir / "expressions.json"
constants_path = artifact_dir / "constants.json"
inference_result_path = artifact_dir / "inference-result.json"
generation_result_path = artifact_dir / "generation-result.json"
generated_model_path = artifact_dir / "generated_fable_2021_model.py"
generated_values_path = artifact_dir / "generated-values.json"
validation_scenario_path = artifact_dir / "validation-scenario.json"
evaluation_report_path = artifact_dir / "evaluation-report.json"

display(Markdown(f"Workbook path: `{workbook_path.relative_to(repo_root)}`"))
display(Markdown(f"Artifact directory: `{artifact_dir.relative_to(repo_root)}`"))

In [ ]:
missing = [path for path in (workbook_path,) if not path.exists()]
ARTIFACTS_READY = not missing

if missing:
    display(Markdown("Missing local artifact(s):"))
    display([str(path.relative_to(repo_root)) for path in missing])
else:
    display(Markdown("Local 2021 workbook is ready."))

## Derive Modelwright output refs

FABLE Pyculator reads the FABLE output tables and selects workbook cells from columns tagged `OUTPUT-*`. Modelwright receives the resulting explicit `output_refs.json` file.

In [ ]:
if ARTIFACTS_READY:
    fable_spec = build_2021_notebook_spec(workbook_path)
    output_refs = sorted(
        {
            row[column_index]
            for table in fable_spec.output_tables
            for column_index, tag in enumerate(table.column_flavour_tags)
            if tag is not None and tag.startswith("OUTPUT-")
            for row in table.cell_refs
        }
    )
    output_refs_path.write_text(json.dumps(output_refs, indent=2), encoding="utf-8")
    display(Markdown(f"Wrote `{len(output_refs):,}` output refs to `{output_refs_path.relative_to(repo_root)}`."))
    display(output_refs[:10])
else:
    output_refs = []
    display(Markdown("Restore the 2021 workbook before deriving output refs."))

## Prepare cached-workbook validation

The validation scenario uses nonblank cached workbook values for the selected output refs.

In [ ]:
def cached_value_kind(value: object) -> str:
    if isinstance(value, bool):
        return "boolean"
    if value is None:
        return "blank"
    if isinstance(value, (int, float)):
        return "number"
    if isinstance(value, str) and value.startswith("#"):
        return "error"
    return "text"


if ARTIFACTS_READY and output_refs:
    from openpyxl import load_workbook

    cached_workbook = load_workbook(workbook_path, data_only=True, read_only=True)
    validation_outputs = []
    for cell_ref in output_refs:
        sheet_name, coordinate = cell_ref.split("!", maxsplit=1)
        value = cached_workbook[sheet_name][coordinate].value
        kind = cached_value_kind(value)
        if kind == "blank":
            continue
        output = {"cell_ref": cell_ref, "kind": kind}
        if kind == "number":
            output["tolerance"] = 1e-9
        validation_outputs.append(output)

    validation_scenario = {
        "scenario_id": "fable-c-2021-freshforge-run",
        "description": "Cached-workbook validation slice derived from FABLE Pyculator OUTPUT-* refs.",
        "source_workbook": str(workbook_path.relative_to(repo_root)),
        "generated_model": str(generated_model_path.relative_to(repo_root)),
        "oracle": {"backend": "cached_workbook"},
        "inputs": [],
        "outputs": validation_outputs,
        "comparison": {"default_numeric_tolerance": 1e-9, "text": "exact", "boolean": "exact"},
    }
    validation_scenario_path.write_text(json.dumps(validation_scenario, indent=2), encoding="utf-8")
    display(Markdown(f"Wrote `{len(validation_outputs):,}` validation outputs to `{validation_scenario_path.relative_to(repo_root)}`."))
else:
    display(Markdown("Restore the 2021 workbook and derive output refs before writing a validation scenario."))

## Declare the FreshForge workflow

This graph uses Modelwright provider nodes for contract inference, model generation, generated-model execution, and cached validation evaluation.

In [ ]:
workflow_document = {
    "workflow": {
        "id": "fable_2021_modelwright_run",
        "name": "FABLE 2021 Modelwright FreshForge run",
        "description": "FreshForge graph for rebuilding the 2021 FABLE generated model.",
    },
    "nodes": [
        {
            "id": "infer_contract",
            "provider": "modelwright.model_infer_contract",
            "outputs": {
                "generated_contract": "generated_contract",
                "formula_expressions": "formula_expressions",
                "input_constants": "input_constants",
            },
            "parameters": {
                "workbook": str(workbook_path.relative_to(repo_root)),
                "module_name": "generated_fable_2021_model",
            },
            "artifacts": {
                "output_refs": str(output_refs_path.relative_to(repo_root)),
                "contract": str(contract_path.relative_to(repo_root)),
                "expressions": str(expressions_path.relative_to(repo_root)),
                "constants": str(constants_path.relative_to(repo_root)),
                "inference_result": str(inference_result_path.relative_to(repo_root)),
            },
        },
        {
            "id": "generate_model",
            "provider": "modelwright.model_generate",
            "needs": ["infer_contract"],
            "inputs": {
                "generated_contract": "infer_contract.generated_contract",
                "formula_expressions": "infer_contract.formula_expressions",
                "input_constants": "infer_contract.input_constants",
            },
            "outputs": {"generated_model": "generated_fable_2021_model"},
            "artifacts": {
                "contract": str(contract_path.relative_to(repo_root)),
                "expressions": str(expressions_path.relative_to(repo_root)),
                "constants": str(constants_path.relative_to(repo_root)),
                "generated_model": str(generated_model_path.relative_to(repo_root)),
                "generation_result": str(generation_result_path.relative_to(repo_root)),
            },
        },
        {
            "id": "execute_model",
            "provider": "modelwright.model_execute",
            "needs": ["generate_model"],
            "inputs": {
                "generated_contract": "infer_contract.generated_contract",
                "generated_model": "generate_model.generated_model",
            },
            "outputs": {"generated_values": "generated_values"},
            "artifacts": {
                "contract": str(contract_path.relative_to(repo_root)),
                "generated_model": str(generated_model_path.relative_to(repo_root)),
                "generated_values": str(generated_values_path.relative_to(repo_root)),
            },
        },
        {
            "id": "evaluate_model",
            "provider": "modelwright.validation_evaluate",
            "needs": ["execute_model"],
            "inputs": {
                "generated_contract": "infer_contract.generated_contract",
                "generated_model": "generate_model.generated_model",
            },
            "outputs": {"validation_report": "validation_report"},
            "parameters": {
                "scenario": str(validation_scenario_path.relative_to(repo_root)),
                "workbook": str(workbook_path.relative_to(repo_root)),
            },
            "artifacts": {
                "contract": str(contract_path.relative_to(repo_root)),
                "generated_model": str(generated_model_path.relative_to(repo_root)),
                "scenario": str(validation_scenario_path.relative_to(repo_root)),
                "evaluation_report": str(evaluation_report_path.relative_to(repo_root)),
            },
        },
    ],
}

workflow_path.write_text(json.dumps(workflow_document, indent=2), encoding="utf-8")
display(Markdown(f"Wrote FreshForge workflow graph to `{workflow_path.relative_to(repo_root)}`."))

## Validate and plan

In [ ]:
try:
    from freshforge.loading import load_workflow
    from freshforge.planning import create_run_plan

    FRESHFORGE_READY = True
except ModuleNotFoundError:
    FRESHFORGE_READY = False
    display(Markdown("FreshForge is not installed in this kernel."))

if FRESHFORGE_READY:
    workflow_spec, diagnostics = load_workflow(workflow_path)
    plan = create_run_plan(workflow_spec, diagnostics=diagnostics) if workflow_spec is not None else None
    display(plan.to_dict() if plan is not None else [diagnostic.to_dict() for diagnostic in diagnostics])
    if plan is None or plan.has_errors:
        raise RuntimeError("FreshForge could not plan the Modelwright workflow graph.")

## Run with FreshForge

The run is gated because the full 2021 build can take several minutes. Review the plan first, then set `RUN_FRESHFORGE = True`.

In [ ]:
RUN_FRESHFORGE = False

if RUN_FRESHFORGE:
    if not ARTIFACTS_READY:
        raise RuntimeError("Restore the 2021 workbook before running the build.")
    from freshforge.execution import run_workflow
    from freshforge.loading import load_workflow

    workflow_spec, diagnostics = load_workflow(workflow_path)
    if workflow_spec is None:
        raise RuntimeError([diagnostic.to_dict() for diagnostic in diagnostics])
    run_result = run_workflow(workflow_spec, diagnostics=diagnostics, workdir=repo_root)
    display(run_result.to_dict())
    if not run_result.ok:
        raise RuntimeError("FreshForge run failed; inspect diagnostics above.")
else:
    display(Markdown("FreshForge run not started. Set `RUN_FRESHFORGE = True` after reviewing the plan."))

## Inspect FABLE Pyculator outputs

After the FreshForge run creates the 2021 generated model, load that exact generated artifact and inspect FABLE output tables/headlines.

In [ ]:
if generated_model_path.exists() and ARTIFACTS_READY:
    generated_model = load_generated_model(
        generated_model_path,
        module_name="fable_pyculator_freshforge_generated_fable_2021",
    )
    result = run_notebook_loop(
        generated_model,
        fable_spec,
        {},
        scenario_name="freshforge-2021",
        output_table_column_flavour_tags="OUTPUT-*",
        include_figures=False,
    )
    display(Markdown(f"Rendered `{len(result.output_tables)}` output tables and `{len(result.headline_frames)}` headline frames."))
    display(result.output_tables["ghg_resultsghg"].head())
else:
    display(Markdown(f"Generated model not found yet: `{generated_model_path.relative_to(repo_root)}`"))